# Colab + Google Drive Learning Cycle Template

This notebook provides a minimal workflow to:

1. Mount Google Drive
2. Prepare repository and dependencies on Colab
3. Run training and prediction using Drive data
4. Confirm output artifacts


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

REPO_DIR = '/content/kyotei_Prediction'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/takajin0114/kyotei_Prediction.git {REPO_DIR}

%cd {REPO_DIR}
!python -m pip install --upgrade pip
!pip install -r requirements.txt


In [ ]:
import os

DRIVE_ROOT = '/content/drive/MyDrive/kyotei_prediction'
DATA_DIR = f'{DRIVE_ROOT}/data/raw'
YEAR_MONTH = '2026-02'
PREDICT_DATE = '2026-02-14'
N_TRIALS = 1
MINIMAL = True

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print('DRIVE_ROOT =', DRIVE_ROOT)
print('DATA_DIR   =', DATA_DIR)


In [ ]:
# Optional: fetch/update raw data directly into Drive.
# Set RUN_FETCH = True when needed.

RUN_FETCH = False

if RUN_FETCH:
    !python -m kyotei_predictor.tools.batch.batch_fetch_all_venues \
      --start-date 2026-02-01 \
      --end-date 2026-02-14 \
      --stadiums ALL \
      --output-data-dir "{DATA_DIR}"


In [ ]:
import subprocess

cmd = [
    'python',
    'scripts/run_colab_learning_cycle.py',
    '--drive-root', DRIVE_ROOT,
    '--data-dir', DATA_DIR,
    '--year-month', YEAR_MONTH,
    '--n-trials', str(N_TRIALS),
    '--predict-date', PREDICT_DATE,
]
if MINIMAL:
    cmd.append('--minimal')

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import json
import os

pred_path = f'outputs/predictions_{PREDICT_DATE}.json'
print('prediction file exists:', os.path.exists(pred_path), pred_path)

if os.path.exists(pred_path):
    with open(pred_path, 'r', encoding='utf-8') as f:
        pred = json.load(f)
    print('prediction_date:', pred.get('prediction_date'))
    print('summary:', pred.get('execution_summary'))
